#Code Cell 1 — Manual workflow vs Pipeline

In [2]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Load dataset
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(1 - data.target, name="target")  # 0=Benign, 1=Malignant

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==================================
# Approach 1: Manual Preprocessing
# ==================================

# Imputation
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

# Model
lr_manual = LogisticRegression(max_iter=2000)
lr_manual.fit(X_train_scaled, y_train)

pred_manual = lr_manual.predict(X_test_scaled)
manual_accuracy = accuracy_score(y_test, pred_manual)

# ==================================
# Approach 2: Using Pipeline
# ==================================

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

pipeline.fit(X_train, y_train)
pred_pipeline = pipeline.predict(X_test)

pipeline_accuracy = accuracy_score(y_test, pred_pipeline)

# ==================================
# Comparison
# ==================================

print("Manual Workflow Accuracy:", manual_accuracy)
print("Pipeline Accuracy:", pipeline_accuracy)

print("\nPredictions Identical:",
      (pred_manual == pred_pipeline).all())

print("\nPipeline Steps:")
print(pipeline.named_steps)

Manual Workflow Accuracy: 0.9649122807017544
Pipeline Accuracy: 0.9649122807017544

Predictions Identical: True

Pipeline Steps:
{'imputer': SimpleImputer(strategy='median'), 'scaler': StandardScaler(), 'classifier': LogisticRegression(max_iter=2000)}


#Code Cell 2 — Pipeline Inspection

In [3]:
# Access individual pipeline steps

print("Imputer Median Values (first 5):")
print(pipeline.named_steps["imputer"].statistics_[:5])

print("\nScaler Means (first 5):")
print(pipeline.named_steps["scaler"].mean_[:5])

print("\nScaler Values (first 5):")
print(pipeline.named_steps["scaler"].scale_[:5])

print("\nLogistic Regression Coefficients (first 5):")
print(pipeline.named_steps["classifier"].coef_[0][:5])

print("\nTotal Pipeline Steps:", len(pipeline.steps))

Imputer Median Values (first 5):
[1.334e+01 1.890e+01 8.618e+01 5.464e+02 9.524e-02]

Scaler Means (first 5):
[1.41660769e+01 1.94176923e+01 9.22158681e+01 6.59578242e+02
 9.59929451e-02]

Scaler Values (first 5):
[3.57514584e+00 4.28593536e+00 2.46899415e+01 3.60022403e+02
 1.42939574e-02]

Logistic Regression Coefficients (first 5):
[0.36115007 0.4822194  0.35315987 0.43995028 0.35062156]

Total Pipeline Steps: 3
